# Titanic: Machine Learning from Disaster bằng PyTorch

Bài thực hành xây dựng **mạng MLP đơn giản bằng PyTorch** để dự đoán hành khách sống sót trên Titanic.

Quy trình được trình bày gần giống tài liệu mẫu:

1. Load Data
2. Data Analysis
3. Data Preprocessing
4. Define Model
5. Fit Model
6. Model Evaluation
7. Tinh chỉnh tham số đơn giản
8. Prediction
9. Submission

> Cần đặt các file `train.csv`, `test.csv`, `gender_submission.csv`
> trong cùng thư mục với notebook.

## 1. Import thư viện

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import wandb

print("PyTorch:", torch.__version__)

ModuleNotFoundError: No module named 'wandb'

## 2. Load Data

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Số hành khách trong train:", len(train))
print("Số hành khách trong test :", len(test))

train.head()

## 3. Data Analysis

In [ ]:
train.info()

In [ ]:
print("Giá trị thiếu trong train:")
print(train.isnull().sum())

In [ ]:
# Số hành khách sống sót / không sống sót
train["Survived"].value_counts().sort_index().plot(kind="bar")
plt.title("Số hành khách sống sót")
plt.xlabel("Survived")
plt.ylabel("Số lượng")
plt.show()

In [ ]:
# Phân bố tuổi
train["Age"].plot(kind="hist", bins=20)
plt.title("Phân bố tuổi")
plt.xlabel("Age")
plt.show()

## 4. Data Preprocessing

Để giữ bài đơn giản giống tài liệu PDF, ta sử dụng các thuộc tính:

- `Pclass`
- `Sex`
- `Age`
- `SibSp`
- `Parch`
- `Fare`
- `Embarked`

Các bước:

- Điền tuổi thiếu bằng tuổi trung bình.
- Điền Fare thiếu bằng giá vé trung bình.
- One-hot encoding cho `Sex`, `Embarked`, `Pclass`.
- Chuẩn hóa các biến số.

In [ ]:
def preprocess_titanic(df):
    data = df.copy()

    # Điền dữ liệu thiếu
    data["Age"] = data["Age"].fillna(data["Age"].mean())
    data["Fare"] = data["Fare"].fillna(data["Fare"].mean())
    data["Embarked"] = data["Embarked"].fillna(data["Embarked"].mode()[0])

    # One-hot encoding
    sex = pd.get_dummies(data["Sex"], prefix="Sex", drop_first=True)
    embark = pd.get_dummies(data["Embarked"], prefix="Embarked", drop_first=True)
    pclass = pd.get_dummies(data["Pclass"], prefix="Pclass", drop_first=True)

    data = pd.concat([data, sex, embark, pclass], axis=1)

    # Bỏ các cột không sử dụng
    data = data.drop(
        ["Pclass", "Sex", "Embarked", "Cabin", "PassengerId", "Name", "Ticket"],
        axis=1
    )

    return data


train_p = preprocess_titanic(train)
test_p = preprocess_titanic(test)

train_p.head()

### Đồng bộ cột giữa train và test

In [ ]:
X = train_p.drop("Survived", axis=1)
y = train_p["Survived"]

# test không có cột Survived
X_test_kaggle = test_p.copy()

# Đảm bảo train và test có cùng cột
X, X_test_kaggle = X.align(
    X_test_kaggle,
    join="left",
    axis=1,
    fill_value=0
)

print("Các cột đầu vào:")
print(X.columns.tolist())

### Chia train / validation

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=1,
    stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test_kaggle_scaled = scaler.transform(X_test_kaggle)

print(X_train.shape, X_val.shape)

### Chuyển sang Tensor

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)

X_kaggle_t = torch.tensor(X_test_kaggle_scaled, dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=32,
    shuffle=True
)

print(X_train_t.shape, y_train_t.shape)

## 5. Define Model

Mạng MLP đơn giản:

**Input → Linear → ReLU → Linear → ReLU → Output**

In [ ]:
class TitanicMLP(nn.Module):
    def __init__(self, input_size, hidden1=32, hidden2=16):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_size, hidden1),
            nn.ReLU(),

            nn.Linear(hidden1, hidden2),
            nn.ReLU(),

            nn.Linear(hidden2, 1)
        )

    def forward(self, x):
        return self.net(x)


model = TitanicMLP(X_train.shape[1])

print(model)

## 6. Fit Model

Dùng:

- Loss: `BCEWithLogitsLoss`
- Optimizer: `Adam`
- Epoch: `100`

W&B chỉ được dùng để theo dõi `loss` và `accuracy` cho đơn giản.

Nếu chưa muốn kết nối W&B, đổi:

```python
WANDB_MODE = "disabled"
```

In [ ]:
WANDB_MODE = "disabled"   # đổi thành "online" nếu muốn dùng W&B

run = wandb.init(
    project="titanic-pytorch",
    config={
        "hidden1": 32,
        "hidden2": 16,
        "learning_rate": 0.001,
        "batch_size": 32,
        "epochs": 100
    },
    mode=WANDB_MODE
)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 100

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Validation
    model.eval()

    with torch.no_grad():
        val_logits = model(X_val_t)
        val_probs = torch.sigmoid(val_logits)
        val_pred = (val_probs >= 0.5).int().numpy().ravel()

    val_acc = accuracy_score(y_val, val_pred)
    avg_loss = total_loss / len(train_loader)

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_loss,
        "val_accuracy": val_acc
    })

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1:3d}/{epochs} - "
            f"Loss: {avg_loss:.4f} - "
            f"Val Acc: {val_acc:.4f}"
        )

wandb.finish()

## 7. Model Evaluation

In [ ]:
model.eval()

with torch.no_grad():
    logits = model(X_val_t)
    probs = torch.sigmoid(logits)
    predictions = (probs >= 0.5).int().numpy().ravel()

print(classification_report(y_val, predictions))

In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_val, predictions))

print("\nAccuracy:")
print(accuracy_score(y_val, predictions))

## 8. Tinh chỉnh tham số mạng đơn giản

Ta thử **3 cấu hình** khác nhau thay vì dùng Grid Search phức tạp.

Các tham số được thử:

- `hidden1`
- `hidden2`
- `learning_rate`

Mục tiêu: chọn cấu hình có Accuracy trên validation cao nhất.

In [ ]:
configs = [
    {"hidden1": 16, "hidden2": 8,  "lr": 0.001},
    {"hidden1": 32, "hidden2": 16, "lr": 0.001},
    {"hidden1": 64, "hidden2": 32, "lr": 0.001}
]

results = []

for cfg in configs:

    m = TitanicMLP(
        input_size=X_train.shape[1],
        hidden1=cfg["hidden1"],
        hidden2=cfg["hidden2"]
    )

    optimizer = torch.optim.Adam(m.parameters(), lr=cfg["lr"])
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(50):
        m.train()

        for xb, yb in train_loader:
            optimizer.zero_grad()

            logits = m(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

    m.eval()

    with torch.no_grad():
        pred = (
            torch.sigmoid(m(X_val_t)) >= 0.5
        ).int().numpy().ravel()

    acc = accuracy_score(y_val, pred)

    results.append({
        "hidden1": cfg["hidden1"],
        "hidden2": cfg["hidden2"],
        "lr": cfg["lr"],
        "accuracy": acc,
        "model": m
    })

result_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != "model"}
    for r in results
])

result_df

In [ ]:
best_result = max(results, key=lambda x: x["accuracy"])

best_model = best_result["model"]

print("Cấu hình tốt nhất:")
print("hidden1 =", best_result["hidden1"])
print("hidden2 =", best_result["hidden2"])
print("lr      =", best_result["lr"])
print("accuracy=", best_result["accuracy"])

## 9. Prediction cho tập test Kaggle

Tập `test.csv` không có nhãn `Survived`.

Ta dùng `best_model` để dự đoán cho toàn bộ 418 hành khách trong tập test.

In [ ]:
best_model.eval()

with torch.no_grad():
    test_logits = best_model(X_kaggle_t)

    test_probabilities = torch.sigmoid(test_logits)

    test_predictions = (
        test_probabilities >= 0.5
    ).int().numpy().ravel()

print(test_predictions[:20])
print("Số dự đoán:", len(test_predictions))

## 10. Submission

File submission cần có đúng 2 cột:

- `PassengerId`
- `Survived`

Ta lấy `PassengerId` trực tiếp từ `test.csv` và dùng kết quả dự đoán của PyTorch.

In [ ]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_predictions
})

submission.head()

In [ ]:
submission.to_csv("submission_pytorch.csv", index=False)

print("Đã tạo file: submission_pytorch.csv")
print("Kích thước:", submission.shape)

### Kiểm tra file submission

In [ ]:
check_submission = pd.read_csv("submission_pytorch.csv")

print(check_submission.head())
print()
print(check_submission.info())

## 11. Kết luận

Quy trình PyTorch trong bài:

**Load Data → Data Analysis → Data Preprocessing → Define Model → Fit Model → Model Evaluation → Tuning → Prediction → Submission**

Điểm khác so với Logistic Regression trong tài liệu mẫu là:

- Mô hình được xây dựng bằng `torch.nn.Module`.
- Huấn luyện bằng vòng lặp PyTorch.
- Dùng `BCEWithLogitsLoss`.
- Dùng `Adam`.
- Dự đoán xác suất bằng `torch.sigmoid`.
- Kết quả cuối cùng được lưu vào `submission_pytorch.csv` để nộp Kaggle.